### Import all packages

In [ ]:
import ikarus as iks
from ikarus import finite_elements, utils, assembler, dirichlet_values

import dune.grid
import dune.functions
from dune.vtk import vtkUnstructuredGridWriter, RangeTypes, FieldInfo

import numpy as np
import scipy as sp
import matplotlib
from scipy.optimize import minimize
import pyvista as pv

### Create grid

In [ ]:
lowerLeft = [-1,-1]
upperRight = [1,1]
elements = [3,3]

grid = dune.grid.structuredGrid(lowerLeft,upperRight,elements)
grid.hierarchicalGrid.globalRefine(0)
grid.plot()

### Add Lagrangian basis

In [ ]:
basisLagrange1 = iks.basis(grid, dune.functions.Power(dune.functions.Lagrange(order=1), 2))
flatBasis = basisLagrange1.flat()
print('We have {} dofs.'.format(len(flatBasis)))
print('We have {} vertices.'.format(grid.size(2)))
print('We have {} elements.'.format(grid.size(0)))

### Initialize load factor

In [ ]:
lambdaLoad = iks.ValueWrapper(3.0)

### Define volume load and boundary loads

In [ ]:
def vL(x,lambdaVal) :
    return np.array([lambdaVal * x[0] * 2, 2 * lambdaVal * x[1] * 0])

def nL(x,lambdaVal) :
    return np.array([lambdaVal * 0, lambdaVal])

neumannVertices = np.zeros(grid.size(2), dtype=bool)
def loadTopEdgePredicate(x):
        return True if x[1] > 1-1e-8 else False

indexSet = grid.indexSet
for v in grid.vertices:
    neumannVertices[indexSet.index(v)]=loadTopEdgePredicate(v.geometry.center)

boundaryPatch = iks.utils.boundaryPatch(grid, neumannVertices)

volumeLoad = iks.finite_elements.volumeLoad2D(vL)
neumannLoad= iks.finite_elements.neumannBoundaryLoad(boundaryPatch, nL)

### Create material

In [ ]:
svk = iks.materials.StVenantKirchhoff(E=1000, nu=0.3)

svkPS = svk.asPlaneStress()

### Create vector of finite elements

In [ ]:
nonLinElastic = iks.finite_elements.nonLinearElastic(svkPS)

fes = []
for e in grid.elements:
    fes.append(iks.finite_elements.makeFE(basisLagrange1, nonLinElastic, volumeLoad, neumannLoad))
    fes[-1].bind(e)

### Create Dirichlet boundary conditions

In [ ]:
dirichletValues = iks.dirichletValues(flatBasis) 

def fixFirstIndex(vec, globalIndex):
        vec[0] = True

def fixAnotherVertex(vec, localIndex, localView):
    localView.index(localIndex)
    vec[1] = True

def fixLeftHandEdge(vec, localIndex, localView, intersection):
    if intersection.geometry.center[1] < -0.9:
        vec[localView.index(localIndex)] = True

dirichletValues.fixBoundaryDOFs(fixFirstIndex)
dirichletValues.fixBoundaryDOFsUsingLocalView(fixAnotherVertex)
dirichletValues.fixBoundaryDOFsUsingLocalViewAndIntersection(fixLeftHandEdge)

### Create assembler

In [ ]:
assembler = iks.assembler.sparseFlatAssembler(fes, dirichletValues)
dRed = np.zeros(assembler.reducedSize())

### Define functions

In [ ]:
def energy(dRedInput):
    feReq = fes[0].createRequirement()
    feReq.insertParameter( lambdaLoad)
    dBig = assembler.createFullVector(dRedInput)
    feReq.insertGlobalSolution( dBig)
    return assembler.scalar(
        feReq, iks.ScalarAffordance.mechanicalPotentialEnergy
    )

def gradient(dRedInput):
    feReq = fes[0].createRequirement()
    feReq.insertParameter(lambdaLoad)
    dBig = assembler.createFullVector(dRedInput)
    feReq.insertGlobalSolution(dBig)
    return assembler.vector(
        feReq, iks.VectorAffordance.forces, iks.DBCOption.Reduced)

def hess(dRedInput):
    feReq = fes[0].createRequirement()
    feReq.insertParameter(lambdaLoad)
    dBig = assembler.createFullVector(dRedInput)
    feReq.insertGlobalSolution(dBig)
    return assembler.matrix(
        feReq, iks.MatrixAffordance.stiffness, iks.DBCOption.Reduced
    ).todense()

### Solve using scipy

In [ ]:
resultd = minimize(energy, x0=dRed, options={"disp": True}, tol=1e-14)
resultd2 = minimize(
    energy, x0=dRed, jac=gradient, options={"disp": True}, tol=1e-14
)
resultd3 = minimize(
    energy,
    method="trust-constr",
    x0=dRed,
    jac=gradient,
    hess=hess,
    options={"disp": True},
)
resultd4 = sp.optimize.root(gradient, jac=hess, x0=dRed, tol=1e-10)

In [ ]:
assert np.allclose(resultd.x, resultd2.x, atol=1e-6)
assert np.allclose(resultd3.x, resultd4.x)
assert np.all(abs(resultd3.grad) < 1e-8)
assert np.all(abs(resultd4.fun) < 1e-8)

In [ ]:
fullDisp = assembler.createFullVector(resultd2.x)
dispFunc = flatBasis.asFunction(fullDisp)

feReq = fes[0].createRequirement()
feReq.insertGlobalSolution(fullDisp)

stressFunc = grid.function(
    lambda e, x: fes[indexSet.index(e)].calculateAt(feReq, x, "PK2Stress")[:]
)

In [ ]:
filename = "Exercise10"
vtkWriter = vtkUnstructuredGridWriter(grid)
vtkWriter.addPointData(dispFunc, name="displacement")
vtkWriter.addPointData(stressFunc, name="stress")
vtkWriter.write(filename)

In [ ]:
def plot(filename):
    # Postprocessing with pyVista (doesnt seem to work within devcontainer)

    mesh = pv.UnstructuredGrid(filename + ".vtu")
    plotter = pv.Plotter(off_screen=True)
    plotter.view_xy()
    plotter.add_mesh(mesh, scalars="displacement", component=2, show_edges=True)
    plotter.show()

In [ ]:
plot(filename)

### Plot here using matplotlib

In [ ]:
import dune.plotting
dune.plotting.plot(solution=dispFunc, gridLines="black")